<a href="https://colab.research.google.com/github/jjkiljanski/biebrza-shrub-encroachment-analysis/blob/main/notebooks/select_representative_series.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Biebrza – Representative Pixel Time Series & Class Trajectories

This notebook:
1. Loads the exported biannual pixel time-series CSV.
2. Reconstructs the same preprocessing used for the Conv1D model.
3. Loads the trained Conv1D model weights.
4. Runs inference on the **test split** to get per-class probabilities.
5. Selects representative examples per class (correct / misclassified / false-positive).
6. Computes class-level mean trajectories.
7. Saves:
   - `representative_examples.csv`
   - `class_trajectories.csv`.

You can then use these CSVs in your Streamlit app.

In [1]:
# Imports

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.utils import shuffle as sk_shuffle
from sklearn.preprocessing import LabelEncoder


## 1. Paths & basic config
Adjust these paths to match your Google Drive / local layout.

In [4]:
from google.colab import drive
drive.mount('/content/drive')

# Exported per-pixel time-series CSV (6 bands, biannual)
data_path = '/content/drive/MyDrive/GEE_Biebrza/biebrza_biannual_pixel_series_6_bands.csv'

# Normalization stats saved from training
norm_stats_path = '/content/drive/MyDrive/GEE_Biebrza/norm_stats_biannual_6bands.csv'

# Trained Conv1D model weights
model_path = '/content/conv1d_best_model.pth'

# Output CSVs for Streamlit
out_examples_csv = '/content/drive/MyDrive/GEE_Biebrza/representative_examples.csv'
out_traj_csv     = '/content/drive/MyDrive/GEE_Biebrza/class_trajectories.csv'

print("Using:")
print("  data_path       :", data_path)
print("  norm_stats_path :", norm_stats_path)
print("  model_path      :", model_path)
print("  examples_csv    :", out_examples_csv)
print("  traj_csv        :", out_traj_csv)


Mounted at /content/drive
Using:
  data_path       : /content/drive/MyDrive/GEE_Biebrza/biebrza_biannual_pixel_series_6_bands.csv
  norm_stats_path : /content/drive/MyDrive/GEE_Biebrza/norm_stats_biannual_6bands.csv
  model_path      : /content/conv1d_best_model.pth
  examples_csv    : /content/drive/MyDrive/GEE_Biebrza/representative_examples.csv
  traj_csv        : /content/drive/MyDrive/GEE_Biebrza/class_trajectories.csv


## 2. Load data & apply class relabeling

In [5]:
# Load full dataset
df = pd.read_csv(data_path)
print('Raw data shape:', df.shape)
print('Columns (first 25):', df.columns.tolist()[:25])

print("\nSample of traj_simpl & numark:")
print(df[['traj_simpl', 'numark']].head())

print('\nUnique traj_simpl values:', df['traj_simpl'].unique())
print('\nNumber of unique numark:', df['numark'].nunique())

# Relabel categories
df['class_str'] = df['traj_simpl']

# Merge wetland_to_shrubs + wetland_to_trees -> wetland_to_woody
df.loc[df['class_str'].isin(['wetland_to_shrubs', 'wetland_to_trees']),
       'class_str'] = 'wetland_to_woody'

target_classes = [
    'wetland_to_woody',
    'shrubs_to_trees',
    'stable_wetland',
    'stable_trees',
    'stable_shrubs',
]

df = df[df['class_str'].isin(target_classes)].copy()

print("\nAfter filtering to target classes, shape:", df.shape)
print("Class counts (full filtered dataset):")
print(df['class_str'].value_counts())


Raw data shape: (142327, 66)
Columns (first 25): ['system:index', 'NBR_1997_1998', 'NBR_1999_2000', 'NBR_2001_2002', 'NBR_2003_2004', 'NBR_2005_2006', 'NBR_2007_2008', 'NBR_2009_2010', 'NBR_2011_2012', 'NBR_2013_2014', 'NBR_2015_2016', 'NDMI_1997_1998', 'NDMI_1999_2000', 'NDMI_2001_2002', 'NDMI_2003_2004', 'NDMI_2005_2006', 'NDMI_2007_2008', 'NDMI_2009_2010', 'NDMI_2011_2012', 'NDMI_2013_2014', 'NDMI_2015_2016', 'NDVI_1997_1998', 'NDVI_1999_2000', 'NDVI_2001_2002', 'NDVI_2003_2004']

Sample of traj_simpl & numark:
      traj_simpl numark
0  stable_shrubs    P84
1  stable_shrubs    P84
2  stable_shrubs    P84
3  stable_shrubs    P84
4  stable_shrubs    P84

Unique traj_simpl values: ['stable_shrubs' 'stable_trees' 'stable_wetland' 'wetland_to_shrubs'
 'shrubs_to_trees' 'wetland_to_trees']

Number of unique numark: 13855

After filtering to target classes, shape: (142327, 67)
Class counts (full filtered dataset):
class_str
stable_wetland      71229
stable_trees        45845
stable_shrubs

## 3. Train/Val/Test split by `numark`

In [6]:
# Shuffle full dataset
df_all = sk_shuffle(df, random_state=42).reset_index(drop=True)

print("\nFinal per-class counts (after shuffle):")
print(df_all['class_str'].value_counts())

# Stratified group split by class_str and numark
train_squares = set()
val_squares = set()
test_squares = set()

rng = np.random.default_rng(123)

for cls in target_classes:
    df_cls = df_all[df_all['class_str'] == cls]
    squares = df_cls['numark'].dropna().unique()
    squares = list(squares)
    rng.shuffle(squares)

    n = len(squares)
    n_train = int(0.6 * n)
    n_val = int(0.2 * n)
    # rest -> test

    train_s = squares[:n_train]
    val_s = squares[n_train:n_train + n_val]
    test_s = squares[n_train + n_val:]

    train_squares.update(train_s)
    val_squares.update(val_s)
    test_squares.update(test_s)

print('Unique train squares:', len(train_squares))
print('Unique val squares  :', len(val_squares))
print('Unique test squares :', len(test_squares))

# Build splits
is_train = df_all['numark'].isin(train_squares)
is_val   = df_all['numark'].isin(val_squares)
is_test  = df_all['numark'].isin(test_squares)

df_train = df_all[is_train].copy()
df_val   = df_all[is_val].copy()
df_test  = df_all[is_test].copy()

print('\nSplit sizes (rows):')
print('Train:', df_train.shape[0])
print('Val  :', df_val.shape[0])
print('Test :', df_test.shape[0])

print('\nPer-class counts in Train:')
print(df_train['class_str'].value_counts())
print('\nPer-class counts in Val:')
print(df_val['class_str'].value_counts())
print('\nPer-class counts in Test:')
print(df_test['class_str'].value_counts())



Final per-class counts (after shuffle):
class_str
stable_wetland      71229
stable_trees        45845
stable_shrubs       18990
shrubs_to_trees      4584
wetland_to_woody     1679
Name: count, dtype: int64
Unique train squares: 8312
Unique val squares  : 2770
Unique test squares : 2773

Split sizes (rows):
Train: 85488
Val  : 28511
Test : 28328

Per-class counts in Train:
class_str
stable_wetland      42793
stable_trees        27474
stable_shrubs       11437
shrubs_to_trees      2762
wetland_to_woody     1022
Name: count, dtype: int64

Per-class counts in Val:
class_str
stable_wetland      14205
stable_trees         9247
stable_shrubs        3815
shrubs_to_trees       908
wetland_to_woody      336
Name: count, dtype: int64

Per-class counts in Test:
class_str
stable_wetland      14231
stable_trees         9124
stable_shrubs        3738
shrubs_to_trees       914
wetland_to_woody      321
Name: count, dtype: int64


## 4. Load normalization stats & define time-series columns

In [7]:
norm_df = pd.read_csv(norm_stats_path)
print("Loaded norm stats with", len(norm_df), "features")
print(norm_df.head())

ts_cols = norm_df['col'].tolist()
mean_vec = norm_df['mean'].values.astype('float32')
std_vec  = norm_df['std'].values.astype('float32')

print("\nNumber of time-series columns:", len(ts_cols))
print("First 12 time-series cols:", ts_cols[:12])

feature_types = sorted({c.split('_')[0] for c in ts_cols})
C = len(feature_types)
T = len(ts_cols) // C

assert len(ts_cols) == C * T, (
    f'Expected len(ts_cols)={len(ts_cols)} to be a multiple of C={C}.'
)

print(f"Feature types: {feature_types}")
print(f"Sequence length T={T}, channels C={C}")


Loaded norm stats with 60 features
             col      mean       std
0  NBR_1997_1998  0.266412  0.051863
1  NBR_1999_2000  0.276463  0.055741
2  NBR_2001_2002  0.310012  0.051301
3  NBR_2003_2004  0.306458  0.045374
4  NBR_2005_2006  0.299410  0.053837

Number of time-series columns: 60
First 12 time-series cols: ['NBR_1997_1998', 'NBR_1999_2000', 'NBR_2001_2002', 'NBR_2003_2004', 'NBR_2005_2006', 'NBR_2007_2008', 'NBR_2009_2010', 'NBR_2011_2012', 'NBR_2013_2014', 'NBR_2015_2016', 'NDMI_1997_1998', 'NDMI_1999_2000']
Feature types: ['NBR', 'NDMI', 'NDVI', 'NIR', 'SWIR1', 'SWIR2']
Sequence length T=10, channels C=6


## 5. Normalize using training stats

In [8]:
def normalize_with_stats(df_in, ts_cols, mean_vec, std_vec):
    df_out = df_in.copy()
    X = df_out[ts_cols].values.astype('float32')
    X_norm = (X - mean_vec) / (std_vec + 1e-6)
    df_out[ts_cols] = X_norm
    return df_out

df_train_norm = normalize_with_stats(df_train, ts_cols, mean_vec, std_vec)
df_val_norm   = normalize_with_stats(df_val,   ts_cols, mean_vec, std_vec)
df_test_norm  = normalize_with_stats(df_test,  ts_cols, mean_vec, std_vec)

print("Normalized splits.")


Normalized splits.


## 6. Label encoding

In [9]:
le = LabelEncoder()
le.fit(target_classes)

print("Label mapping:")
for cls, idx in zip(le.classes_, range(len(le.classes_))):
    print(f"  {cls} -> {idx}")

df_train_norm['label_idx'] = le.transform(df_train_norm['class_str'])
df_val_norm['label_idx']   = le.transform(df_val_norm['class_str'])
df_test_norm['label_idx']  = le.transform(df_test_norm['class_str'])

num_classes = len(le.classes_)
print("\nnum_classes:", num_classes)


Label mapping:
  shrubs_to_trees -> 0
  stable_shrubs -> 1
  stable_trees -> 2
  stable_wetland -> 3
  wetland_to_woody -> 4

num_classes: 5


## 7. Dataset for inference on the test set

In [10]:
class PixelTimeSeriesDataset(Dataset):
    def __init__(self, df, ts_cols, label_col, C):
        X = df[ts_cols].values.astype(np.float32)
        N = X.shape[0]
        T = len(ts_cols) // C
        X = X.reshape(N, T, C)
        self.X = X
        self.y = df[label_col].values.astype(np.int64)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

test_ds = PixelTimeSeriesDataset(df_test_norm, ts_cols, 'label_idx', C)
test_loader = DataLoader(test_ds, batch_size=512, shuffle=False)

print("Test dataset size:", len(test_ds))


Test dataset size: 28328


## 8. Load trained Conv1D model

In [11]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

class Conv1DClassifier(nn.Module):
    def __init__(self, seq_len, num_classes, in_channels):
        super().__init__()
        self.seq_len = seq_len
        self.conv1 = nn.Conv1d(in_channels=in_channels, out_channels=32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(in_channels=32,        out_channels=64, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc   = nn.Linear(64, num_classes)

    def forward(self, x):
        x = x.permute(0, 2, 1)   # [B, C, T]
        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))
        x = self.pool(x).squeeze(-1)  # [B, 64]
        logits = self.fc(x)
        return logits

model = Conv1DClassifier(seq_len=T, num_classes=num_classes, in_channels=C)
state_dict = torch.load(model_path, map_location=device)
model.load_state_dict(state_dict)
model.to(device)
model.eval()

print("Model loaded successfully.")


Using device: cpu
Model loaded successfully.


## 9. Run inference on the test set

In [12]:
all_probs = []
all_y_true = []
all_y_pred = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        logits = model(X_batch)
        probs = F.softmax(logits, dim=1)
        preds = probs.argmax(dim=1)

        all_probs.append(probs.cpu().numpy())
        all_y_true.append(y_batch.cpu().numpy())
        all_y_pred.append(preds.cpu().numpy())

all_probs = np.vstack(all_probs)
all_y_true = np.concatenate(all_y_true)
all_y_pred = np.concatenate(all_y_pred)

print("Inference done. all_probs shape:", all_probs.shape)


Inference done. all_probs shape: (28328, 5)


## 10. Attach predictions & probabilities to `df_test_norm`

In [13]:
df_test_ext = df_test_norm.copy()

df_test_ext['true_idx'] = all_y_true
df_test_ext['pred_idx'] = all_y_pred
df_test_ext['pred_class_str'] = le.inverse_transform(all_y_pred)

prob_true = all_probs[np.arange(len(all_y_true)), all_y_true]
prob_pred = all_probs[np.arange(len(all_y_pred)), all_y_pred]
df_test_ext['prob_true'] = prob_true
df_test_ext['prob_pred'] = prob_pred

for i, cls in enumerate(le.classes_):
    df_test_ext[f'prob_{cls}'] = all_probs[:, i]

print("Extended test dataframe shape:", df_test_ext.shape)


Extended test dataframe shape: (28328, 78)


## 11. Select representative examples per class

In [14]:
def select_examples_for_class(df, cls_name, n_correct=10, n_true_mis=10, n_false_pos=10):
    df_true = df[df['class_str'] == cls_name]

    df_correct = df_true[df_true['pred_class_str'] == cls_name]
    df_correct = df_correct.sort_values('prob_true', ascending=False).head(n_correct).copy()
    df_correct['example_type'] = 'correct_highconf'

    df_true_mis = df_true[df_true['pred_class_str'] != cls_name]
    df_true_mis = df_true_mis.sort_values('prob_true', ascending=True).head(n_true_mis).copy()
    df_true_mis['example_type'] = 'true_misclassified'

    df_false_pos = df[(df['pred_class_str'] == cls_name) & (df['class_str'] != cls_name)]
    df_false_pos = df_false_pos.sort_values('prob_pred', ascending=False).head(n_false_pos).copy()
    df_false_pos['example_type'] = 'false_positive'

    return pd.concat([df_correct, df_true_mis, df_false_pos], axis=0)


example_rows = []
for cls_name in le.classes_:
    df_cls_examples = select_examples_for_class(df_test_ext, cls_name,
                                                n_correct=10, n_true_mis=10, n_false_pos=10)
    example_rows.append(df_cls_examples)

df_examples = pd.concat(example_rows, axis=0).reset_index(drop=False).rename(columns={'index': 'orig_row_idx'})

df_examples['example_id'] = (
    df_examples['class_str'] + '_' +
    df_examples['example_type'] + '_' +
    df_examples.index.astype(str)
)

prob_cols = [c for c in df_examples.columns if c.startswith('prob_')]
meta_cols = ['example_id', 'class_str', 'pred_class_str', 'example_type', 'orig_row_idx']
optional_meta = [c for c in ['numark', 'pixel_id'] if c in df_examples.columns]

cols_to_save = meta_cols + optional_meta + ts_cols + prob_cols

df_examples_to_csv = df_examples[cols_to_save]
df_examples_to_csv.to_csv(out_examples_csv, index=False)
print("\nSaved representative examples to:", out_examples_csv)
print("Shape:", df_examples_to_csv.shape)



Saved representative examples to: /content/drive/MyDrive/GEE_Biebrza/representative_examples.csv
Shape: (150, 74)


## 12. Class-level trajectories

In [15]:
def normalize_full(df_all, ts_cols, mean_vec, std_vec):
    df_out = df_all.copy()
    X = df_out[ts_cols].values.astype('float32')
    X_norm = (X - mean_vec) / (std_vec + 1e-6)
    df_out[ts_cols] = X_norm
    return df_out

df_all_norm = normalize_full(df_all, ts_cols, mean_vec, std_vec)
print("df_all_norm shape:", df_all_norm.shape)

def make_class_trajectories(df_in, ts_cols, class_col='class_str'):
    rows = []
    for cls_name, df_cls in df_in.groupby(class_col):
        for col in ts_cols:
            parts = col.split('_')
            feature = parts[0]
            time_label = '_'.join(parts[1:])
            vals = df_cls[col].values
            if len(vals) == 0:
                continue
            rows.append({
                'class_str': cls_name,
                'feature': feature,
                'time_label': time_label,
                'mean': float(np.mean(vals)),
                'std':  float(np.std(vals)),
                'n':    int(len(vals)),
            })
    return pd.DataFrame(rows)

traj_df = make_class_trajectories(df_all_norm, ts_cols, class_col='class_str')
traj_df.to_csv(out_traj_csv, index=False)
print("\nSaved class trajectories to:", out_traj_csv)
print("Shape:", traj_df.shape)


df_all_norm shape: (142327, 67)

Saved class trajectories to: /content/drive/MyDrive/GEE_Biebrza/class_trajectories.csv
Shape: (300, 6)
